In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
import os
import sys

project_path=os.path.join(os.getcwd(),"..")

sys.path.append(project_path)


In [ ]:
from utils.transformation import reusable


# bronze to silver

In [ ]:
df_user_obj=reusable()



### 1.dim_artist

In [ ]:
df_artist=spark.readStream.format("CloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storagedpproject.dfs.core.windows.net/DimArtist/checkpoint/")\
    .load("abfss://bronze@storagedpproject.dfs.core.windows.net/DimArtist")

In [ ]:
df_artist=df_user_obj.dropColumns(df_artist,['_rescued_data'])
df_artist=df_artist.dropDuplicates(['artist_id'])

In [ ]:
df_artist.writeStream\
    .format("delta")\
    .option("checkpointLocation","abfss://silver@storagedpproject.dfs.core.windows.net/DimArtist/checkpoint/")\
    .trigger(once=True)\
    .option("path","abfss://silver@storagedpproject.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotifi.silver.DimArtist")


### 2.DimDate


In [ ]:
df_dimdate=spark.readStream\
        .format("CloudFiles")\
        .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@storagedpproject.dfs.core.windows.net/DimDate/checkpoint/")\
        .load("abfss://bronze@storagedpproject.dfs.core.windows.net/DimDate")
        

In [ ]:
df_dimdate=df_user_oj.dropColumn(df_dimdate,'_rescued_data')

In [ ]:
df_dimdate.writeStream\
    .format("delta")\
    .option("checkPointLocation","abfss://silver@storagedpproject.dfs.core.windows.net/DimDate/checkpoint/")\
    .trigger(once=True)\
    .option("path","abfss://silver@storagedpproject.dfs.core.windows.net/DimDate/data")\
    .toTable("")

### 3.dim_track

In [ ]:
df_track=spark.readStream\
        .format("CloudFiles")\
        .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@storagedpproject.dfs.core.windows.net/DimTrack/checkpoint/")\
        .load("abfss://bronze@storagedpproject.dfs.core.windows.net/DimTrack")

In [ ]:
df_track=df_user_obj.dropColumns(df_track,['_rescued_data'])
df_track=df_track.dropDuplicates(['track_id'])

df_track=df_track.withColumn('durationFlag',when(col('duration_sec')<150 ,'low').when(col('duration_sec')<300,'medium').otherwise('high'))

df_track=df_track.withColumn('track_name',regexp_replace(col('track_name'),"-"," "))


In [ ]:
df_track.writeStream\
    .format("delta")\
    .option("checkPointLocation","abfss://silver@storagedpproject.dfs.core.windows.net/DimTrack/checkpoint/")\
        .trigger(once=True)\
            .option("path","abfss://silver@storagedpproject.dfs.core.windows.net/DimTrack/data")\
            .toTable("spotifi.silver.DimTrack")


### .4 FactStream

In [ ]:
df_factStream=spark.readStream.fromat("CloudFiles")\
        .option("CloudFiles.Format","parquet")\
        .option("ColudFiles.SchemaLocation","abfss://silver@storagedpproject.dfs.core.windows.net/FactStream/checkpoint/")\
        .load("abfss://bronze@storagedpproject.dfs.core.windows.net/FactStream")

In [ ]:
df_factStream=df_user_obj.dropColumns(df_factStream,['_rescued_data'])

In [ ]:
df_factStream.writeStream\
        .format("delta")\
        .option("checkpointLocation","abfss://silver@storagedpproject.dfs.core.windows.net/FactStream/checkpoint/")\
        .trigger(once=True)\
        .option("path","abfss://silver@storagedpproject.dfs.core.windows.net/FactStream/data")\
        .toTable("spotifi.silver.FactStream")

### .5 DimUser

In [ ]:
df_DimUser=spark.readStream\
    .format("CloudFiles")\
    .option("ColudFiles.format","parquet")
    .option('ColudFiles.schemaLocation',"abfss://silver@storagedpproject.dfs.core.windows.net/DimUser/checkpoint/")\
    .load("abfss://bronze@storagedpproject.dfs.core.windows.net/DimUser")

In [ ]:
df_DimUser=df_DimUser.df_user_obj.dropColumns(df_DimUser,['_rescued_data'])
df_DimUser=df_DimUser.dropDuplicates(['user_id'])

In [ ]:
df_DimUser.writeStream\
    .format("delta")\
    .option("checkPointLocation","abfss://silver@storagedpproject.dfs.core.windows.net/DimUser/checkpoint/")\
        .trigger(once=True)\
            .option("path","abfss://silver@storagedpproject.dfs.core.windows.net/DimUser/data")\
            .toTable("spotifi.silver.DimUser")